In [16]:
import numpy as np
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    stream=sys.stdout,
)

logger = logging.getLogger(__name__)

In [28]:
def build_vocab(text) -> tuple[dict[str, int], int]:
    """Build a vocabulary from the input text and return a token table and its size.
    
    Vocab:
    A vocabulary is a mapping of unique words in the text to their corresponding indices. This is created using
    a pre-defined corpus of text, where each unique word is assigned a unique index.
    
    """
    words = text.split()
    token_table = {word: i for i, word in enumerate(set(words))}
    return token_table, len(token_table)

In [29]:
def tokenize(text: str, vocab: dict[str, int]) -> list[int]:
    """Tokenize the input text using the provided vocabulary.
    
    Tokenization:
    Tokenization is basically retrieving the ID or index of each word in a text input using an already built vocabulary.
    """
    tokens = [vocab[word] for word in text.split()]
    return tokens

In [4]:
def initialize_embeddings(vocab_size, embedding_dim=300):
    # Initialize a token embedding with random values
    E = np.random.rand(vocab_size, embedding_dim)
    return E

# test
vocab_size = 3
embedding_dim = 2

print(initialize_embeddings(vocab_size, embedding_dim))

[[0.58365941 0.92619883]
 [0.54318168 0.22728342]
 [0.90165491 0.75292967]]


In [30]:
# Example of embedding initialization
text = "hello world hello"

print("Input Text:", text)
vocab, vocab_size = build_vocab(text)
print("Vocabulary:", vocab)
print("Vocabulary Size:", vocab_size)
tokens = tokenize(text, vocab)
print("Tokens:", tokens)
embedding_table = initialize_embeddings(vocab_size, embedding_dim)
print("Embedding Table:\n", embedding_table)

Input Text: hello world hello
Vocabulary: {'world': 0, 'hello': 1}
Vocabulary Size: 2
Tokens: [1, 0, 1]
Embedding Table:
 [[0.39022586 0.3823576 ]
 [0.10543876 0.32913061]]


# Tokenization

## Byte Pair Encoding (BPE)
Byte Pair Encoding (BPE) is a data compression technique that iteratively replaces the most frequent pair of bytes in a sequence with a single, unused byte. This method is commonly used in natural language processing (NLP) for tokenization, where it helps to reduce the vocabulary size while maintaining the ability to represent a wide range of words and subwords. BPE is particularly effective in handling out-of-vocabulary words and can improve the performance of language models by allowing them to learn subword representations. The process involves creating a vocabulary of subword units based on the frequency of byte pairs,which can then be used for encoding and decoding text data.

### How BPE Works
1. **Initialization**: Start with a vocabulary that includes all individual characters (or bytes) in the text corpus and a special end-of-word token.
2. **Counting Pairs**: Count the frequency of all adjacent byte pairs in the text.
3. **Merging**: Identify the most frequent byte pair and merge it into a new token. This new token is added to the vocabulary.
4. **Repeat**: Repeat the counting and merging process until a specified vocabulary size is reached or no more pairs can be merged.

In [41]:
text = """2.6 Why Applications Are Operating-System Specifi 77
ELF FORMAT
Linux provides various commands to identify and evaluate ELF files. For
example, the file command determines a file type. If main.o is an object
file, and main is an executable file, the command
file main.o
will report that main.o is an ELF relocatable file, while the command
file main
will report that main is an ELF executable. ELF files are divided into a number
of sections and can be evaluated using the readelf command.
executable files. One piece of information in the ELF file for executable files is
the program’s entry point, which contains the address of the first instruction
to be executed when the program runs. Windows systems use the Portable
Executable (PE) format, and macOS uses the Mach-O format.
2.6 Why Applications Are Operating-System Specific
Fundamentally, applications compiled on one operating system are not exe-
cutable on other operating systems. If they were, the world would be a better
place, and our choice of what operating system to use would depend on utility
and features rather than which applications were available.
Based on our earlier discussion, we can now see part of the problem—each
operating system provides a unique set of system calls. System calls are part of
the set of services provided by operating systems for use by applications. Even
if system calls were somehow uniform, other barriers would make it difficult
for us to execute application programs on different operating systems. But if
you have used multiple operating systems, you may have used some of the
same applications on them. How is that possible?
An application can be made available to run on multiple operating systems
in one of three ways:
1. The application can be written in an interpreted language (such as Python
or Ruby) that has an interpreter available for multiple operating systems.
The interpreter reads each line of the source program, executes equivalent
instructions on the native instruction set, and calls native operating sys-
tem calls. Performance suffers relative to that for native applications, and
the interpreter provides only a subset of each operating system’s features,
possibly limiting the feature sets of the associated applications.
2. The application can be written in a language that includes a virtual
machine containing the running application. The virtual machine is part
of the language’s full RTE. One example of this method is Java. Java has an
RTE that includes a loader, byte-code verifier, and other components that
load the Java applicatbion into the Java virtual machine. This RTE has been"""

In [7]:
import collections

words = ['gallahad', 'galadriel', 'galadriel']
c = collections.Counter(words)
print(c)

Counter({'galadriel': 2, 'gallahad': 1})


In [ ]:
import collections

def get_stats(words: list[list[str]]) -> collections.Counter:
    pairs = collections.Counter()
    for word in words:
        for i in range(len(word) - 1):
            pair = (word[i], word[i+1])
            pairs[pair] += 1
    return pairs

def merge_words(words: list[list[str]], pair: tuple[str, str]) -> list[list[str]]:
    new_symbol = "".join(pair)
    new_words = []
    for word in words:
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i+1]) == pair:
                new_word.append(new_symbol)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_words.append(new_word)
    return new_words

def byte_pair_encoding(text: str, vocab_size: int):
    # 1. Initialize words
    words = [list(word) + ['<EOW>'] for word in text.split()]
    
    # 2. Initialize Vocab with IDs
    # Start with unique characters
    unique_chars = sorted(list(set(char for word in words for char in word)))
    vocab = {char: i for i, char in enumerate(unique_chars)}
    
    # 3. 
    while len(vocab) < vocab_size:
        pair_freq = get_stats(words)
        if not pair_freq:
            break
            
        best_pair = pair_freq.most_common(1)[0][0]
        new_symbol = "".join(best_pair)
        
        # Assign the next available integer ID
        if new_symbol not in vocab:
            vocab[new_symbol] = len(vocab)
            
        words = merge_words(words, best_pair)

    return vocab, words

vocab, tokens = byte_pair_encoding(text, 1000)

tokens

[['2.6<EOW>'],
 ['Why<EOW>'],
 ['Applications<EOW>'],
 ['Are<EOW>'],
 ['Operating-System<EOW>'],
 ['Specifi<EOW>'],
 ['77<EOW>'],
 ['ELF<EOW>'],
 ['FORMAT<EOW>'],
 ['Linux<EOW>'],
 ['provides<EOW>'],
 ['various<EOW>'],
 ['commands<EOW>'],
 ['to<EOW>'],
 ['identify<EOW>'],
 ['and<EOW>'],
 ['evaluate<EOW>'],
 ['ELF<EOW>'],
 ['files.<EOW>'],
 ['For<EOW>'],
 ['example,<EOW>'],
 ['the<EOW>'],
 ['file<EOW>'],
 ['command<EOW>'],
 ['determines<EOW>'],
 ['a<EOW>'],
 ['file<EOW>'],
 ['type.<EOW>'],
 ['If<EOW>'],
 ['main.o<EOW>'],
 ['is<EOW>'],
 ['an<EOW>'],
 ['object<EOW>'],
 ['file,<EOW>'],
 ['and<EOW>'],
 ['main<EOW>'],
 ['is<EOW>'],
 ['an<EOW>'],
 ['executable<EOW>'],
 ['file,<EOW>'],
 ['the<EOW>'],
 ['command<EOW>'],
 ['file<EOW>'],
 ['main.o<EOW>'],
 ['will<EOW>'],
 ['report<EOW>'],
 ['that<EOW>'],
 ['main.o<EOW>'],
 ['is<EOW>'],
 ['an<EOW>'],
 ['ELF<EOW>'],
 ['relocatable<EOW>'],
 ['file,<EOW>'],
 ['while<EOW>'],
 ['the<EOW>'],
 ['command<EOW>'],
 ['file<EOW>'],
 ['main<EOW>'],
 ['will<EOW

# Embedding Initialization
Embedding initialization refers to the process of setting the initial values of the embedding vectors in a neural network. Proper initialization is crucial for the training of neural networks, as it can affect the convergence speed and overall performance of the model. Common methods for embedding initialization include:
1. **Random Initialization**: Embedding vectors are initialized with random values, often drawn from a uniform or normal distribution.
2. **Pre-trained Embeddings**: Using pre-trained embeddings (e.g., Word2Vec, GloVe) that have been trained on large corpora can provide a good starting point for the model, as they capture semantic relationships between words.
3. **Xavier/Glorot Initialization**: This simply samples a scalar from a uniform distribution where the standard deviation is a function of the number of input and output units in the weight tensor. The formula for Xavier initialization is:
$$\text{limit} = \sqrt{\frac{6}{\text{fan in} + \text{fan out}}}$$


**fan in**: The number of input units in the weight tensor.

**fan out**: The number of output units in the weight tensor.

In this notebook, we will use Xavier initialization.

In [ ]:
# Xavier Initialization
def xavier_initialization(vocab_size, embedding_dim):
    limit = np.sqrt(6 / (vocab_size + embedding_dim))
    E = np.random.uniform(-limit, limit, (vocab_size, embedding_dim))
    return E

# Encoding and Decoding (Transformer)

Source: [Deep Dive into Transformers](https://www.datacamp.com/tutorial/how-transformers-work)

In the transformer architecture, the encoder uses an operation called "self-attention", which is *a way* to
